In [3]:
"""
Module 1 — Data Pipeline
========================
Parses LSV filenames, fits LSV curves, loads EDX composition data,
and merges everything into a single unified DataFrame ready for the GP model.
 
Input:
    - 3x EDX CSV files  (one per library)
    - 966x LSV CSV files (342 per library)
 
Output:
    - unified_data.csv  (966 rows × columns: library, area, x, y, Au%, Ir%, k0, alpha, j_lim)
"""
 
import re
import warnings
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import curve_fit
 
# Suppress scipy curve_fit warnings for failed fits
warnings.filterwarnings("ignore", category=RuntimeWarning)
print("All imports done successfully")

All imports done successfully


In [4]:
def analytical_lsv(e: np.ndarray, j_lim: float, k_0: float, alpha: float) -> np.ndarray:
    """
    Analytical LSV model from Anderson et al. 2023.
    Combines Butler-Volmer kinetics with mass transport in one equation.
 
    Parameters
    ----------
    e     : potential array (V vs RHE)
    j_lim : limiting current density (A/cm²) — the plateau current
    k_0   : standard rate constant (cm/s) — intrinsic catalytic activity
    alpha : charge transfer coefficient (dimensionless, 0–1)
 
    Returns
    -------
    current density (A/cm²), always negative for cathodic reaction
    """
    c_b = 1 / 10_000   # bulk concentration of reactant (mol/cm³)
    F   = 96485         # Faraday constant (C/mol)
    R   = 8.314472      # gas constant (J/mol/K)
    T   = 298.15        # temperature (K), assumed 25°C
    f   = F / (R * T)  # thermal voltage factor = 38.924 V⁻¹
    n   = 2             # electrons transferred per HER event (2H⁺ + 2e⁻ → H₂)
    e_0 = 0             # equilibrium potential vs RHE = 0 V by definition
 
    # Mass transport velocity (cm/s) — how fast reactant arrives at surface
    # derived from the limiting current: j_lim = n * F * c_b * m
    m = j_lim / (n * F * c_b)
 
    # Butler-Volmer exponential terms
    cathodic_term = np.exp(-alpha * f * (e - e_0))          # drives reduction
    anodic_term   = np.exp((1 - alpha) * f * (e - e_0))     # drives oxidation
 
    # Combined model numerator and denominator
    # When k_0 is large → m/k_0 ≈ 0 → pure Butler-Volmer behavior
    # When k_0 is small → m/k_0 dominates → mass transport limited
    nom   = cathodic_term
    denom = m / k_0 + cathodic_term + anodic_term
 
    return -j_lim * nom / denom
 
 
def fit_lsv(lsv: pd.DataFrame) -> pd.Series | None:
    """
    Fit a single LSV curve to extract [j_lim, k_0, alpha].
 
    Strategy:
    - Keep only cathodic (negative) current data
    - Use the most negative current as initial guess for j_lim
    - Bound parameters physically to help curve_fit converge
    - Return None if fit fails (excluded from downstream analysis)
 
    Parameters
    ----------
    lsv : DataFrame with columns:
          'Potential vs. RHE [V]' and 'Current density [A/cm^2]'
 
    Returns
    -------
    pd.Series with [j_lim, k_0, alpha] or None if fit failed
    """
    x = lsv['Potential vs. RHE [V]'].values
    y = lsv['Current density [A/cm^2]'].values
 
    # Keep only cathodic region (negative current = HER happening)
    # Positive current would be oxidation — not relevant for HER fitting
    mask = y <= 0
    x, y = x[mask], y[mask]
 
    # Need at least 10 points to fit 3 parameters reliably
    if len(x) < 10:
        return None
 
    # Most negative current ≈ limiting current plateau
    ymin = np.min(y)
 
    # Initial parameter guesses
    # j_lim ≈ |ymin|, k_0 = small positive, alpha = 0.5 (symmetric)
    p0 = [-ymin, 0.001, 0.5]
 
    # Physical bounds:
    # j_lim must be close to observed plateau (within 10%)
    # k_0 must be positive and < 0.1 cm/s (reasonable for HER)
    # alpha must be between 0 and 1
    bounds = (
        [0.9 * (-ymin), 0,   0],   # lower bounds
        [1.1 * (-ymin), 0.1, 1]    # upper bounds
    )
 
    try:
        opt, _ = curve_fit(analytical_lsv, x, y, p0=p0, bounds=bounds, maxfev=10000)
        return pd.Series(opt, index=['j_lim', 'k_0', 'alpha'])
    except (RuntimeError, ValueError):
        # RuntimeError: curve_fit did not converge
        # ValueError:   input data was invalid
        return None

In [5]:
def parse_lsv_filename(filepath: Path) -> dict | None:
    """
    Extract metadata from an LSV filename.
 
    Expected format:
        Au-Ir-Rh_Au-rich_SECCM_area_1_x_-22_50_y_-40_50_LSV.csv
 
    Note: underscores replace dots and minus signs in x,y values
    because some operating systems don't allow dots in filenames.
    We reconstruct the decimal values by joining the number parts.
 
    Returns
    -------
    dict with keys: library, area, x, y
    or None if filename does not match expected pattern
    """
    name = filepath.stem  # filename without .csv extension
 
    # Regex breakdown:
    # Au-Ir-Rh_         → ternary system prefix (ignored)
    # (Au-rich|Ir-rich|Rh-rich) → capture library name
    # _SECCM_area_(\d+) → capture area number (integer)
    # _x_([-\d_]+)      → capture x coordinate (may have minus and decimals as underscores)
    # _y_([-\d_]+)      → capture y coordinate
    pattern = r'Au-Ir-Rh_(Au-rich|Ir-rich|Rh-rich)_SECCM_area_(\d+)_x_([-\d_]+)_y_([-\d_]+)_LSV'
    match = re.search(pattern, name)
 
    if not match:
        return None
 
    library = match.group(1)   # e.g. "Au-rich"
    area    = int(match.group(2))  # e.g. 1
 
    # Reconstruct x coordinate: "_-22_50" → "-22.50"
    # The last underscore-separated segment is the decimal part
    x_raw = match.group(3)  # e.g. "-22_50"
    y_raw = match.group(4)  # e.g. "-40_50"
 
    def reconstruct_coordinate(raw: str) -> float:
        """Convert underscore-encoded coordinate back to float."""
        # Handle negative sign: "-22_50" → parts = ["-22", "50"]
        # Split on last underscore to separate integer and decimal parts
        last_underscore = raw.rfind('_')
        integer_part = raw[:last_underscore]   # e.g. "-22"
        decimal_part = raw[last_underscore+1:] # e.g. "50"
        return float(f"{integer_part}.{decimal_part}")
 
    x = reconstruct_coordinate(x_raw)
    y = reconstruct_coordinate(y_raw)
 
    return {'library': library, 'area': area, 'x': x, 'y': y}

In [8]:

def process_all_lsvs(lsv_folder: str | Path) -> pd.DataFrame:
    """
    Scan a folder for all LSV CSV files, parse their filenames,
    fit each curve, and return a DataFrame of results.
 
    Parameters
    ----------
    lsv_folder : path to folder containing all LSV CSV files
                 (can contain files from all 3 libraries mixed together)
 
    Returns
    -------
    DataFrame with columns: library, area, x, y, j_lim, k_0, alpha
    One row per successfully fitted LSV.
    """
    lsv_folder = Path(lsv_folder)
    lsv_files  = list(lsv_folder.glob("*_LSV.csv"))
 
    print(f"Found {len(lsv_files)} LSV files in {lsv_folder}")
 
    records = []
 
    for filepath in lsv_files:
        # Step 1: parse filename to get metadata
        meta = parse_lsv_filename(filepath)
        if meta is None:
            print(f"  [SKIP] Could not parse filename: {filepath.name}")
            continue
 
        # Step 2: load the LSV data
        try:
            lsv_data = pd.read_csv(filepath)
        except Exception as e:
            print(f"  [SKIP] Could not read file {filepath.name}: {e}")
            continue
 
        # Step 3: fit the LSV curve
        fit_result = fit_lsv(lsv_data)
        if fit_result is None:
            print(f"  [SKIP] Fit failed for {filepath.name}")
            continue
 
        # Step 4: combine metadata + fit results into one record
        record = {**meta, **fit_result.to_dict()}
        records.append(record)
 
    # Assemble into DataFrame
    df = pd.DataFrame(records)
    print(f"Successfully fitted {len(df)} LSVs")
 
    return df

In [9]:

def load_edx_files(edx_file_map: dict[str, str | Path]) -> pd.DataFrame:
    """
    Load EDX composition files for all three libraries and combine them.
 
    Parameters
    ----------
    edx_file_map : dict mapping library name to EDX CSV filepath
        e.g. {
            'Au-rich': 'data/edx/Au-Ir-Rh_Au-rich_EDX.csv',
            'Ir-rich': 'data/edx/Au-Ir-Rh_Ir-rich_EDX.csv',
            'Rh-rich': 'data/edx/Au-Ir-Rh_Rh-rich_EDX.csv',
        }
 
    Returns
    -------
    DataFrame with columns: library, area, Au [at.%], Ir [at.%], Rh [at.%]
    """
    frames = []
 
    for library, filepath in edx_file_map.items():
        df = pd.read_csv(filepath)
 
        # Rename 'Area' column to lowercase for consistency
        df = df.rename(columns={'Area': 'area'})
 
        # Tag with library name so we can distinguish area 1 across libraries
        df['library'] = library
 
        frames.append(df)
        print(f"  Loaded EDX for {library}: {len(df)} areas")
 
    edx_combined = pd.concat(frames, ignore_index=True)
    print(f"Total EDX rows: {len(edx_combined)}")
 
    return edx_combined

In [10]:

def build_unified_dataset(lsv_df: pd.DataFrame, edx_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge LSV fit results with EDX composition data on (library, area).
    Apply feature engineering for the GP model.
 
    Feature engineering:
    - Drop Rh [at.%]: redundant since Au + Ir + Rh = 100 always
      (keeping all 3 would cause perfect multicollinearity in the GP)
    - Add log_k0: log-transform of k_0 to handle orders-of-magnitude variation
      (GP assumes Gaussian noise — log makes k_0 distribution more Gaussian)
 
    Parameters
    ----------
    lsv_df : output of process_all_lsvs()
    edx_df : output of load_edx_files()
 
    Returns
    -------
    Unified DataFrame with 966 rows (or fewer if some fits failed/were excluded)
    """
    # Merge on composite key (library, area)
    # Using inner join: only keep areas present in BOTH datasets
    merged = pd.merge(lsv_df, edx_df, on=['library', 'area'], how='inner')
 
    print(f"\nMerged dataset: {len(merged)} rows")
    print(f"Libraries: {merged['library'].value_counts().to_dict()}")
 
    # Drop Rh — it's always 100 - Au - Ir, so it's fully redundant
    # Keeping it would make the GP input space rank-deficient
    if 'Rh [at.%]' in merged.columns:
        merged = merged.drop(columns=['Rh [at.%]'])
 
    # Log-transform k_0
    # k_0 spans ~2 orders of magnitude → log compresses to ~4 unit range
    # This is needed for the GP's Gaussian noise assumption to hold
    merged['log_k0'] = np.log(merged['k_0'])
 
    # Reorder columns for clarity
    col_order = ['library', 'area', 'x', 'y',
                 'Au [at.%]', 'Ir [at.%]',
                 'j_lim', 'k_0', 'log_k0', 'alpha']
    merged = merged[col_order]
 
    # Sort by library then area for reproducibility
    merged = merged.sort_values(['library', 'area']).reset_index(drop=True)
 
    return merged

In [11]:
def run_pipeline(lsv_folder: str, edx_file_map: dict, output_path: str = "unified_data.csv") -> pd.DataFrame:
    """
    Run the complete data pipeline end-to-end.
 
    Parameters
    ----------
    lsv_folder   : folder containing all LSV CSV files
    edx_file_map : dict of {library_name: edx_csv_path}
    output_path  : where to save the unified CSV
 
    Returns
    -------
    unified DataFrame saved to output_path
    """
    print("=" * 60)
    print("STEP 1: Processing LSV files")
    print("=" * 60)
    lsv_df = process_all_lsvs(lsv_folder)
 
    print("\n" + "=" * 60)
    print("STEP 2: Loading EDX composition files")
    print("=" * 60)
    edx_df = load_edx_files(edx_file_map)
 
    print("\n" + "=" * 60)
    print("STEP 3: Merging and feature engineering")
    print("=" * 60)
    unified = build_unified_dataset(lsv_df, edx_df)
 
    # Save to CSV — this is the interface file read by the GPR environment
    unified.to_csv(output_path, index=False)
    print(f"\nSaved unified dataset to: {output_path}")
 
    print("\n" + "=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(unified.describe())
 
    return unified
 

In [12]:

if __name__ == "__main__":
 
    # Edit these paths to match your folder structure
    LSV_FOLDER = "data/lsvs"
 
    EDX_FILES = {
        'Au-rich': 'data/edx/Au-Ir-Rh_Au-rich_EDX.csv',
        'Ir-rich': 'data/edx/Au-Ir-Rh_Ir-rich_EDX.csv',
        'Rh-rich': 'data/edx/Au-Ir-Rh_Rh-rich_EDX.csv',
    }
 
    unified_data = run_pipeline(
        lsv_folder   = LSV_FOLDER,
        edx_file_map = EDX_FILES,
        output_path  = "data/unified_data.csv"
    )
 
    print("\nFirst 5 rows:")
    print(unified_data.head())

STEP 1: Processing LSV files
Found 0 LSV files in data\lsvs
Successfully fitted 0 LSVs

STEP 2: Loading EDX composition files


FileNotFoundError: [Errno 2] No such file or directory: 'data/edx/Au-Ir-Rh_Au-rich_EDX.csv'

In [15]:
"""
Module 1 — Data Pipeline (Simplified)
======================================
Merges pre-fitted LSV parameters with EDX composition data.
Fitting is already done — LSV_fit_parameters.csv has all 966 results.

Input:
    - LSV_fit_parameters.csv     (966 rows: Library, Area, i_lim, k0, alpha)
    - Au-Ir-Rh_Au-rich_EDX.csv  (342 rows: Area, Au%, Ir%, Rh%)
    - Au-Ir-Rh_Ir-rich_EDX.csv
    - Au-Ir-Rh_Rh-rich_EDX.csv

Output:
    - unified_data.csv           (966 rows, ready for GPR)
"""

import numpy as np
import pandas as pd
from pathlib import Path


# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 — Load pre-fitted LSV parameters
# ─────────────────────────────────────────────────────────────────────────────

def load_lsv_parameters(filepath: str) -> pd.DataFrame:
    """
    Load the pre-fitted LSV parameters and rename columns to clean names.
    """
    df = pd.read_csv(filepath)

    df = df.rename(columns={
        'Library'        : 'library',
        'Area'           : 'area',
        'i_lim [A/cm^2]' : 'j_lim',
        'k^0 [cm/s]'    : 'k0',
        'alpha [a.u.]'   : 'alpha',
    })

    print(f"  Loaded LSV parameters: {len(df)} rows")
    print(f"  Libraries: {df['library'].value_counts().to_dict()}")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 — Load EDX composition files
# ─────────────────────────────────────────────────────────────────────────────

def load_edx_files(edx_file_map: dict) -> pd.DataFrame:
    """
    Load EDX CSV files for all three libraries and combine them.
    Tags each row with library name to create unique (library, area) key.
    """
    frames = []

    for library, filepath in edx_file_map.items():
        df = pd.read_csv(filepath)
        df = df.rename(columns={'Area': 'area'})
        df['library'] = library
        frames.append(df)
        print(f"  Loaded EDX {library}: {len(df)} areas")

    edx = pd.concat(frames, ignore_index=True)
    print(f"  Total EDX rows: {len(edx)}")
    return edx


# ─────────────────────────────────────────────────────────────────────────────
# STEP 3 — Merge and feature engineering
# ─────────────────────────────────────────────────────────────────────────────

def build_unified_dataset(lsv_df: pd.DataFrame, edx_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge LSV parameters with EDX compositions on (library, area).
    Then apply feature engineering for the GP model.

    Feature engineering:
    - Drop Rh [at.%]  : Au + Ir + Rh = 100 always, so Rh is redundant.
                        Keeping it causes multicollinearity in the GP kernel.
    - Add log_k0      : k0 spans ~40x range. Log transform makes the
                        distribution approximately Gaussian for the GP.

    GP inputs  → ['Au [at.%]', 'Ir [at.%]']
    GP outputs → ['log_k0', 'alpha']
    """
    merged = pd.merge(lsv_df, edx_df, on=['library', 'area'], how='inner')
    print(f"\n  After merge: {len(merged)} rows")
    print(f"  Per library: {merged['library'].value_counts().to_dict()}")

    # Drop redundant Rh column
    merged = merged.drop(columns=['Rh [at.%]'])

    # Log-transform k0
    merged['log_k0'] = np.log(merged['k0'])

    # Clean column order
    merged = merged[[
        'library', 'area',
        'Au [at.%]', 'Ir [at.%]',    # GP inputs
        'k0', 'log_k0', 'alpha',      # GP outputs
        'j_lim',                       # kept for reference
    ]]

    merged = merged.sort_values(['library', 'area']).reset_index(drop=True)
    return merged


# ─────────────────────────────────────────────────────────────────────────────
# USER INPUT HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def prompt_file_path(prompt_text: str, must_exist: bool = True) -> str:
    """
    Ask the user to type a file path and validate it exists.
    Strips surrounding quotes in case the user drags-and-drops the file
    (Windows and macOS sometimes wrap paths in quotes).

    Keeps asking until a valid path is provided.
    """
    while True:
        raw = input(prompt_text).strip().strip('"').strip("'")
        path = Path(raw)
        if must_exist and not path.exists():
            print(f"  [ERROR] File not found: {raw}")
            print(f"          Please check the path and try again.\n")
        else:
            return str(path)


def prompt_output_path() -> str:
    """
    Ask the user where to save the unified CSV.
    If they press Enter without typing, use the default path.
    """
    default = "unified_data.csv"
    raw = input(f"\nWhere should unified_data.csv be saved? [default: {default}]\n> ").strip().strip('"').strip("'")
    if raw == "":
        return default
    # If user gave a directory, append filename
    path = Path(raw)
    if path.is_dir():
        return str(path / "unified_data.csv")
    return str(path)


def collect_user_inputs() -> tuple[str, dict, str]:
    """
    Interactively collect all file paths from the user.

    Returns
    -------
    lsv_params_path : str         path to LSV_fit_parameters.csv
    edx_file_map    : dict        {library_name: edx_csv_path}
    output_path     : str         where to save unified_data.csv
    """
    print("\n" + "=" * 55)
    print("MODULE 1 — DATA PIPELINE")
    print("=" * 55)
    print("Please provide the paths to your data files.")
    print("Tip: you can drag and drop files into the terminal.\n")

    # ── LSV fit parameters ────────────────────────────────────────────────────
    print("─── LSV Fit Parameters ─────────────────────────────────")
    lsv_params_path = prompt_file_path(
        "Path to LSV_fit_parameters.csv:\n> "
    )
    print(f"  OK: {lsv_params_path}\n")

    # ── EDX files (one per library) ───────────────────────────────────────────
    print("─── EDX Composition Files ──────────────────────────────")
    edx_file_map = {}
    for library in ['Au-rich', 'Ir-rich', 'Rh-rich']:
        path = prompt_file_path(
            f"Path to EDX file for {library} library:\n> "
        )
        edx_file_map[library] = path
        print(f"  OK: {path}\n")

    # ── Output path ───────────────────────────────────────────────────────────
    print("─── Output ─────────────────────────────────────────────")
    output_path = prompt_output_path()
    print(f"  Will save to: {output_path}\n")

    return lsv_params_path, edx_file_map, output_path


# ─────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(
    lsv_params_path : str,
    edx_file_map    : dict,
    output_path     : str,
) -> pd.DataFrame:
    """
    Run the full pipeline: load → merge → feature engineering → save.
    """
    print("\n" + "=" * 55)
    print("STEP 1: Loading LSV fit parameters")
    print("=" * 55)
    lsv_df = load_lsv_parameters(lsv_params_path)

    print("\n" + "=" * 55)
    print("STEP 2: Loading EDX composition files")
    print("=" * 55)
    edx_df = load_edx_files(edx_file_map)

    print("\n" + "=" * 55)
    print("STEP 3: Merging + feature engineering")
    print("=" * 55)
    unified = build_unified_dataset(lsv_df, edx_df)

    # Save to CSV — this file is the input to all downstream modules
    unified.to_csv(output_path, index=False)

    print("\n" + "=" * 55)
    print("DONE — SUMMARY")
    print("=" * 55)
    print(f"  Saved to : {output_path}")
    print(f"  Shape    : {unified.shape}")
    print()
    print(unified[['Au [at.%]', 'Ir [at.%]', 'log_k0', 'alpha']].describe().round(4))
    print()
    print("First 5 rows:")
    print(unified.head())

    return unified


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Collect paths interactively from the user
    lsv_params_path, edx_file_map, output_path = collect_user_inputs()

    # Run the pipeline with the provided paths
    data = run_pipeline(lsv_params_path, edx_file_map, output_path)



MODULE 1 — DATA PIPELINE
Please provide the paths to your data files.
Tip: you can drag and drop files into the terminal.

─── LSV Fit Parameters ─────────────────────────────────


Path to LSV_fit_parameters.csv:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\SECCM_dataset\LSV_fit_parameters.csv


  OK: C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\SECCM_dataset\LSV_fit_parameters.csv

─── EDX Composition Files ──────────────────────────────


Path to EDX file for Au-rich library:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\EDX_dataset\Au-Ir-Rh_Au-rich_EDX.csv


  OK: C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\EDX_dataset\Au-Ir-Rh_Au-rich_EDX.csv



...

  OK: C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\EDX_dataset\Au-Ir-Rh_Ir-rich_EDX.csv



Path to EDX file for Rh-rich library:
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\EDX_dataset\Au-Ir-Rh_Rh-rich_EDX.csv


  OK: C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\EDX_dataset\Au-Ir-Rh_Rh-rich_EDX.csv

─── Output ─────────────────────────────────────────────



Where should unified_data.csv be saved? [default: unified_data.csv]
>  C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data


  Will save to: C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\unified_data.csv


STEP 1: Loading LSV fit parameters
  Loaded LSV parameters: 966 rows
  Libraries: {'Ir-rich': 322, 'Au-rich': 322, 'Rh-rich': 322}

STEP 2: Loading EDX composition files
  Loaded EDX Au-rich: 342 areas
  Loaded EDX Ir-rich: 342 areas
  Loaded EDX Rh-rich: 342 areas
  Total EDX rows: 1026

STEP 3: Merging + feature engineering

  After merge: 966 rows
  Per library: {'Ir-rich': 322, 'Au-rich': 322, 'Rh-rich': 322}

DONE — SUMMARY
  Saved to : C:\Users\devshri\Documents\python_projects\datasets-requirements\Datasets-pySECCM\Processed_data\unified_data.csv
  Shape    : (966, 8)

       Au [at.%]  Ir [at.%]    log_k0     alpha
count   966.0000   966.0000  966.0000  966.0000
mean     34.9958    38.1740   -5.1857    0.2944
std      21.4245    23.1233    0.6228    0.0180
min       3.8900     6.0700   -8.0325    0.2371
25%      16.4900    18.0225   -5.5349    0.28